In [ ]:
import pandas as pd, json

prediction_df = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/src/stages/pattern_classification/ensemble_classification_predictions.csv')
prediction_df.fillna('None Type', inplace=True)

description_json = json.load(open('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/predicted_clusters_verification_results_with_descriptions.json'))

In [ ]:
def get_code_from_pattern(file):
    file = "/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2/"+file
    with open(file,'r') as f:
        return f.read()
    
def get_code_from_communities(file):
    file = "/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/repo_callgraph_clusters/"+file
    with open(file,'r') as f:
        return f.read()

def get_file(file):
    file = file.split('/')[-1]
    file = "/".join(file.rsplit("-", 1))
    return file
def get_file_description(file):
    file = file.split('/')[-1]
    file = "/".join(file.rsplit("-", 1))
    return description_json[file]['code_summary'] if file in description_json else 'No Description Found'

prediction_df['code summary'] = prediction_df['file'].apply(get_file_description)
prediction_df['file'] = prediction_df['file'].apply(lambda x: "https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/"+get_file(x))
# prediction_df.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/src/stages/pattern_classification/ensemble_classification_predictions_with_descriptions.csv', index=False)

In [ ]:
prediction_df

In [ ]:

prediction_communities = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/src/stages/pattern_classification/predictions_ensemble.csv')
prediction_communities.fillna('None Type',inplace=True)

def get_file_description(file):
    return description_json[file]['code_summary'] if file in description_json else 'No Description Found'

prediction_communities.drop('true_pattern',inplace=True,axis=1,errors='ignore')
prediction_communities['code summary'] = prediction_communities['file'].apply(get_file_description)
prediction_communities['file'] = prediction_communities['file'].apply(lambda x: "https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/"+x)
prediction_communities['already_verified'] = (
    prediction_communities['file'].isin(prediction_df['file'])
)
prediction_communities[prediction_communities['code summary']!='No Description Found'].to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/ensemble_predicted_labels/predicted_labels_for_communities.csv',index=False)

In [ ]:
df = prediction_communities[~prediction_communities['already_verified']]
df = df[df['code summary']!='No Description Found']


# total samples wanted
N = 70


# compute samples per pattern (proportional, at least 1 if possible)
pattern_counts = df['second_level_model_class_rr'].value_counts(normalize=True)
allocations = (pattern_counts * N).round().astype(int)


# fix rounding issues to get exactly N
while allocations.sum() != N:
    diff = N - allocations.sum()
    if diff > 0:
        allocations.iloc[0] += diff
    else:
        idx = allocations[allocations > 1].index[0]
        allocations.loc[idx] -= 1


# stratified sampling
samples = []
for pattern, k in allocations.items():
    subset = df[df['second_level_model_class_rr'] == pattern]
    samples.append(subset.sample(n=min(k, len(subset)), random_state=42))


stratified_sample = pd.concat(samples).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
stratified_sample.to_csv('results/randomly_picked_70-dec-12.csv',index=False)

In [ ]:
df.iloc[0]

### Combine Description + Embedding + Verified Pattern

In [15]:
import pandas as pd,json
verified_communities = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/inputs/AI Pattern Project_ Results - Verified Communities.csv')
verified_communities['Code file'] = verified_communities['Code file'].apply(lambda x:x.replace('https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/',''))
verified_communities['Label'] = verified_communities['Label'].fillna("none")

code_description = json.load(open('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/predicted_clusters_verification_results_with_descriptions.json'))

embeddings = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding.csv')
embeddings.drop(columns=['pattern'],inplace=True)

In [16]:
dataset = embeddings.copy()
dataset = dataset.merge(
    verified_communities[['Code file', 'Label']],
    left_on='file',
    right_on='Code file',
    how='left'
)

dataset = dataset.rename(columns={'Label': 'verified_pattern'})
dataset = dataset.drop(columns=['Code file'])
dataset['verified'] = dataset['verified_pattern'].notnull()
dataset['code_summary'] = dataset['file'].apply(lambda x: code_description.get(x,{'code_summary':None})['code_summary'])
dataset=dataset[dataset['code_summary'].notnull()]
dataset['predicted_pattern'] = None

dataset['file'] = dataset['file'].apply(lambda x: "https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/"+x)

important_cols = ['file', 'verified_pattern', 'predicted_pattern', 'code_summary']
other_cols = [c for c in dataset.columns if c not in important_cols]

dataset = dataset[important_cols + other_cols]

In [17]:
dataset.verified_pattern.value_counts()

verified_pattern
none                                                                                            64
Classical Models                                                                                47
Preprocessing Text and Numerical Data                                                           34
Tool Use for LLMs                                                                               27
LLM based Multimodal Generative Prompting                                                       26
Retrieval Augmented Generation(RAG) Optimization for LLMs                                       20
Modular LLM Agent Architectures                                                                 20
Model Abstraction Pattern                                                                       16
LLM Results Evaluation                                                                          13
Enhanced User Intent Comprehension with LLMs                                                

In [18]:
dataset.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/label-studio-ml-backend/my_ml_backend/data/dataset.csv', index=False)